# Experiência do usuário e QoE do dump Argus

## Escopo e atraso conhecido

Estas tabelas NPAW/Youbora são agregadas por hora x dimensão e não expõem IDs
individuais. Cada consulta usa uma janela UTC explícita `[start, end)`. Há um
atraso estrutural de aproximadamente **28 horas**: a cobertura mais recente pode
estar vazia ou parcial por desenho do pipeline, e não é evidência de ausência de
incidente em tempo real.

In [1]:
from dump_explorer.charts import (
    category_stack,
    geo_heatmap,
    impact_scatter,
    qoe_timeline,
)
from dump_explorer.config import load_settings
from dump_explorer.database import create_db_engine
from dump_explorer.qoe import (
    NPAW_DELAY_HOURS,
    load_device_qoe,
    load_error_categories,
    load_geo_qoe,
    load_live_active,
    load_live_qoe,
    load_player_qoe,
)

# derive a safe UTC epoch-ms window from signal_bucket observed bounds
from dump_explorer.quality import load_source_coverage
import pandas as pd

settings = load_settings()
engine = create_db_engine(settings)

# Obter janela segura a partir da cobertura de dados
s = load_source_coverage(engine)
if s.empty or s['first_ts'].dropna().empty:
    print("⚠️  source_coverage vazio. Usando valores padrão.")
    start_ms = 0
    end_ms = 86_400_000
else:
    start_ms = int(s['first_ts'].dropna().min().value // 10**6)
    end_ms = int(s['last_ts'].dropna().max().value // 10**6) + 1
    print(f"Intervalo de dados: {pd.to_datetime(start_ms, unit='ms', utc=True)} a {pd.to_datetime(end_ms-1, unit='ms', utc=True)}")

Intervalo de dados: 2026-07-10 18:45:00+00:00 a 2026-08-05 23:45:00+00:00


## 1. Players, dispositivos e geografia

`(desconhecido)` é uma dimensão ausente explicitamente marcada, não removida do
ranking. A taxa de erro é `NULL` quando sessões é zero. Dispositivos usam pelo
menos 30 sessões **na consulta SQL**; o ranking não representa a cardinalidade
bruta. O heatmap UF x ISP usa somente agregados locais, sem mapa ou ativos externos.

In [2]:
player_qoe = load_player_qoe(engine, start_ms, end_ms)
device_qoe = load_device_qoe(engine, start_ms, end_ms, min_sessions=30)
geo_qoe = load_geo_qoe(engine, start_ms, end_ms)
display(player_qoe)
display(device_qoe)
display(geo_qoe)
impact_scatter(player_qoe, dimension="player").show()
impact_scatter(device_qoe, dimension="device").show()
geo_heatmap(geo_qoe).show()

,player,sessions,errors,error_rate_pct,dimension_missing
0,,89005514.0,10509665.0,11.807881,True
1,clappr-web,269498556.0,8353672.0,3.099709,False
2,Media3,211630026.0,2794003.0,1.320230,False
3,clappr-web-tvs,264247327.0,2555473.0,0.967076,False
4,ios,93615364.0,680723.0,0.727149,False
...,...,...,...,...,...
56,web-webos_8k_hdr,17624.0,15.0,0.085111,False
57,roku_4k,28017.0,13.0,0.046400,False
58,web-mirakulo_8k_hdr,458.0,0.0,0.000000,False
59,web-hisense_4k,133.0,0.0,0.000000,False


,device,sessions,errors,error_rate_pct,dimension_missing
0,TCL|Smart TV,31759535.0,1987512.0,6.258001,False
1,Google|Nexus 5X,1959806.0,1902333.0,97.067414,False
2,?|Generic Android Mobile,103226422.0,1807130.0,1.750647,False
3,Apple|iPhone,53498864.0,1777787.0,3.323037,False
4,Microsoft|?,103183178.0,1730628.0,1.677239,False
...,...,...,...,...,...
4750,samsung|UN49NU7100,30.0,0.0,0.000000,False
4751,samsung|UN55BU8000GXZD,30.0,0.0,0.000000,False
4752,samsung|UN60AU8000GXZD,30.0,0.0,0.000000,False
4753,samsung|UN60BU8000GXZD,30.0,0.0,0.000000,False


,state_province,isp,sessions,errors,error_rate_pct,dimension_missing
0,SP,?,158121145.0,730057.0,0.461707,True
1,?,Claro NET,102943823.0,410503.0,0.398764,True
2,RJ,?,83928548.0,403751.0,0.481065,True
3,?,Vivo,87196237.0,401804.0,0.460804,True
4,MG,?,58648036.0,267445.0,0.456017,True
...,...,...,...,...,...,...
920,?,Netlink Ltda.,69.0,0.0,0.000000,True
921,?,Uninet,69.0,0.0,0.000000,True
922,?,wlinks.com.br,67.0,0.0,0.000000,True
923,?,Viber Telecom Servicos E Telecomunicacao Ltda,66.0,0.0,0.000000,True


## 2. QoE ao vivo e audiência

Selecione sinais explicitamente. `bufferratio` é **ratio**, `jointime` é em
**segundos** e latência é em **milissegundos**; as unidades não compartilham eixo.
`startup_errors` fica separado de erros durante a reprodução. `active_sessions`
é uma contagem de sessões **concorrentes**, não acumulada; `joins_in_bucket`
significa novas entradas no bucket de 15 minutos.

In [3]:
signals = ["pfc1"]  # selecione explicitamente um ou mais sinais

if not signals:
    print("⚠️  signals está vazia. Adicione nomes de sinais para visualizar dados QoE ao vivo.")
else:
    live_qoe = load_live_qoe(engine, signals, start_ms, end_ms)
    live_active = load_live_active(engine, signals, start_ms, end_ms)
    latest_qoe_timestamp = live_active["timestamp"].max() if not live_active.empty else None
    
    if latest_qoe_timestamp:
        print(
            f"Timestamp NPAW mais recente: {latest_qoe_timestamp}; "
            f"cobertura pode atrasar cerca de {NPAW_DELAY_HOURS} horas."
        )
    else:
        print(f"⚠️  Nenhum dado NPAW encontrado. Cobertura pode estar atrasada (~{NPAW_DELAY_HOURS} horas).")
    
    if not live_qoe.empty:
        display(live_qoe)
        try:
            qoe_timeline(live_qoe, metric="startup_errors").show()
            qoe_timeline(live_qoe, metric="bufferratio_avg").show()
            qoe_timeline(live_qoe, metric="jointime_s_avg").show()
        except Exception as e:
            print(f"❌ Erro ao renderizar timelines QoE: {e}")
    else:
        print(f"⚠️  Nenhum dado QoE encontrado para os sinais: {signals}")
    
    if not live_active.empty:
        display(live_active)
        try:
            qoe_timeline(live_active, metric="active_sessions").show()
        except Exception as e:
            print(f"❌ Erro ao renderizar timeline de sessões ativas: {e}")
    else:
        print(f"⚠️  Nenhum dado de sessões ativas encontrado para os sinais: {signals}")

Timestamp NPAW mais recente: 2026-08-05 23:45:00+00:00; cobertura pode atrasar cerca de 28 horas.


,sinal,sessions,errors,startup_errors,latency_ms_avg,latency_ms_p50,latency_ms_p95,bufferratio_avg,jointime_s_avg,dropped_frames_avg,stalls_avg,timestamp,error_rate_pct,dimension_missing,bufferratio_unit,jointime_unit,latency_unit
0,pfc1,42122,2028,158,8086.979156,4800.0,20420.0,1.653689,4.401567,142.653483,0.012097,2026-07-22 00:00:00+00:00,4.814586,False,ratio,s,ms
1,pfc1,3916,48,42,1417.266599,0.0,9878.0,0.600315,4.792258,1.769663,0.004651,2026-07-22 01:00:00+00:00,1.225741,False,ratio,s,ms
2,pfc1,963,29,29,958.804777,0.0,0.0,0.762033,4.591034,3.787124,0.004496,2026-07-22 02:00:00+00:00,3.011423,False,ratio,s,ms
3,pfc1,200,21,21,1095.000000,0.0,15000.0,0.734430,5.204505,56.260000,0.005825,2026-07-22 03:00:00+00:00,10.500000,False,ratio,s,ms
4,pfc1,104,27,27,836.538462,0.0,12000.0,0.529375,3.535337,3.240385,0.005212,2026-07-22 04:00:00+00:00,25.961538,False,ratio,s,ms
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,pfc1,541,14,14,535.674677,0.0,0.0,0.731730,5.610072,5.674677,0.005887,2026-08-05 19:00:00+00:00,2.587800,False,ratio,s,ms
356,pfc1,1396,21,12,977.255731,0.0,12000.0,0.761052,5.551165,35.109599,0.004868,2026-08-05 20:00:00+00:00,1.504298,False,ratio,s,ms
357,pfc1,36948,2585,93,6548.049231,0.0,18325.0,2.264836,5.113800,429.441296,0.017037,2026-08-05 21:00:00+00:00,6.996319,False,ratio,s,ms
358,pfc1,148051,8091,445,8983.537247,7846.0,17838.0,5.402537,4.249959,309.300950,0.047087,2026-08-05 22:00:00+00:00,5.465009,False,ratio,s,ms


,sinal,active_sessions,active_with_error,joins_in_bucket,latency_ms_p50,latency_ms_p95,bufferratio_avg,jointime_s_avg,dropped_frames_avg,timestamp,dimension_missing,active_sessions_semantic,joins_in_bucket_semantic,bufferratio_unit,jointime_unit,latency_unit
0,pfc1,70485,3419,18291,0.0,15000.0,0.515377,5.746560,347.153196,2026-07-22 00:00:00+00:00,False,concurrent,new_entries,ratio,s,ms
1,pfc1,73188,3371,16054,0.0,15000.0,0.492124,5.715457,325.906597,2026-07-22 00:15:00+00:00,False,concurrent,new_entries,ratio,s,ms
2,pfc1,49204,1752,5646,0.0,13749.0,0.468960,6.083317,293.408077,2026-07-22 00:30:00+00:00,False,concurrent,new_entries,ratio,s,ms
3,pfc1,14595,509,2131,0.0,12000.0,0.439246,6.434538,313.122302,2026-07-22 00:45:00+00:00,False,concurrent,new_entries,ratio,s,ms
4,pfc1,8784,302,1404,0.0,12000.0,0.418165,6.541214,323.747609,2026-07-22 01:00:00+00:00,False,concurrent,new_entries,ratio,s,ms
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1381,pfc1,63143,5518,26467,0.0,21543.0,1.018765,5.643671,678.368956,2026-08-05 22:45:00+00:00,False,concurrent,new_entries,ratio,s,ms
1382,pfc1,50963,3878,24948,0.0,20630.0,0.648274,6.010976,738.334164,2026-08-05 23:00:00+00:00,False,concurrent,new_entries,ratio,s,ms
1383,pfc1,57830,4477,33865,0.0,19473.0,0.926403,5.836049,712.064240,2026-08-05 23:15:00+00:00,False,concurrent,new_entries,ratio,s,ms
1384,pfc1,66145,5122,28163,0.0,19767.0,0.863781,5.713503,698.997853,2026-08-05 23:30:00+00:00,False,concurrent,new_entries,ratio,s,ms


## 3. Categorias semânticas de erro

Live e VOD são carregados e visualizados separadamente; não existe taxa única
misturando os dois. `top_names` preserva o texto bruto e recebe uma flag quando
o JSON está inválido.

In [4]:
live_categories = load_error_categories(engine, "live", start_ms, end_ms)
vod_categories = load_error_categories(engine, "vod", start_ms, end_ms)
display(live_categories)
display(vod_categories)
category_stack(live_categories).show()
category_stack(vod_categories).show()

,kind,category,errors,top_names,timestamp,top_names_parsed,top_names_parse_error
0,live,acesso,30090,"[{""name"": ""login-required"", ""errors"": 21468}, ...",2026-07-22 00:00:00+00:00,"[{'name': 'login-required', 'errors': 21468}, ...",False
1,live,entrega,5756,"[{""name"": ""connection-error"", ""errors"": 1980},...",2026-07-22 00:00:00+00:00,"[{'name': 'connection-error', 'errors': 1980},...",False
2,live,origem,5185,"[{""name"": ""Unexpected audio track timestamp di...",2026-07-22 00:00:00+00:00,[{'name': 'Unexpected audio track timestamp di...,False
3,live,outros,24,"[{""name"": ""104"", ""errors"": 8}, {""name"": ""102"",...",2026-07-22 00:00:00+00:00,"[{'name': '104', 'errors': 8}, {'name': '102',...",False
4,live,player,1104,"[{""name"": ""playback-type-renderer-error"", ""err...",2026-07-22 00:00:00+00:00,"[{'name': 'playback-type-renderer-error', 'err...",False
...,...,...,...,...,...,...,...
1795,live,acesso,16119,"[{""name"": ""login-required"", ""errors"": 8939}, {...",2026-08-05 23:00:00+00:00,"[{'name': 'login-required', 'errors': 8939}, {...",False
1796,live,entrega,6020,"[{""name"": ""connection-error"", ""errors"": 2145},...",2026-08-05 23:00:00+00:00,"[{'name': 'connection-error', 'errors': 2145},...",False
1797,live,origem,9813,"[{""name"": ""Unexpected audio track timestamp di...",2026-08-05 23:00:00+00:00,[{'name': 'Unexpected audio track timestamp di...,False
1798,live,outros,27,"[{""name"": ""102"", ""errors"": 11}, {""name"": ""104""...",2026-08-05 23:00:00+00:00,"[{'name': '102', 'errors': 11}, {'name': '104'...",False


,kind,category,errors,top_names,timestamp,top_names_parsed,top_names_parse_error
0,vod,acesso,27113,"[{""name"": ""video-not-found"", ""errors"": 8253}, ...",2026-07-22 00:00:00+00:00,"[{'name': 'video-not-found', 'errors': 8253}, ...",False
1,vod,entrega,3080,"[{""name"": ""connection-error"", ""errors"": 887}, ...",2026-07-22 00:00:00+00:00,"[{'name': 'connection-error', 'errors': 887}, ...",False
2,vod,origem,23,"[{""name"": ""100"", ""errors"": 14}, {""name"": ""Unex...",2026-07-22 00:00:00+00:00,"[{'name': '100', 'errors': 14}, {'name': 'Unex...",False
3,vod,outros,24,"[{""name"": ""invalid-source"", ""errors"": 7}, {""na...",2026-07-22 00:00:00+00:00,"[{'name': 'invalid-source', 'errors': 7}, {'na...",False
4,vod,player,2312,"[{""name"": ""Tizen:playback-native-error"", ""erro...",2026-07-22 00:00:00+00:00,"[{'name': 'Tizen:playback-native-error', 'erro...",False
...,...,...,...,...,...,...,...
1794,vod,acesso,25643,"[{""name"": ""user-not-authorized"", ""errors"": 767...",2026-08-05 23:00:00+00:00,"[{'name': 'user-not-authorized', 'errors': 767...",False
1795,vod,entrega,3021,"[{""name"": ""connection-error"", ""errors"": 1013},...",2026-08-05 23:00:00+00:00,"[{'name': 'connection-error', 'errors': 1013},...",False
1796,vod,origem,145,"[{""name"": ""Unexpected audio track timestamp di...",2026-08-05 23:00:00+00:00,[{'name': 'Unexpected audio track timestamp di...,False
1797,vod,outros,35,"[{""name"": ""globoplay-sdk-playback:timeout"", ""e...",2026-08-05 23:00:00+00:00,"[{'name': 'globoplay-sdk-playback:timeout', 'e...",False
